| **Chain / Retriever**               | **Returns Citations** | **Method / Output Key**                          | **Status**             | **Docs Link**                                                                               |
| ----------------------------------- | --------------------- | ------------------------------------------------ | ---------------------- | ------------------------------------------------------------------------------------------- |
| `RetrievalQAWithSourcesChain`       | ✅ Yes                 | `result['answer']`, `result['sources']`          | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/retrieval-qa-with-sources) |
| `ConversationalRetrievalChain`      | ✅ Yes                 | `result['answer']`, `result['source_documents']` | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/chat_vector_db)            |
| `MultiRetrievalQAChain`             | ✅ Yes                 | Depends on sub-chains used                       | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/multi_retrieval_qa)        |
| `VectorDBQAWithSourcesChain`        | ✅ Yes                 | `result['answer']`, `result['sources']`          | ✅ (but legacy)         | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/vector-db-qa)              |
| `Tool` using RetrievalQAWithSources | ✅ Yes                 | `result['answer']`, `result['sources']`          | **Stable** (via Agent) | [🔗 Tools Docs](https://docs.langchain.com/docs/modules/agents/tools/custom_tools)          |
| `RetrievalQA` (basic chain)         | ❌ No                  | Only `answer`                                    | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/popular/retrieval-qa)              |
| `RefineDocumentsChain`              | ❌ No                  | Only `answer`                                    | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/document/refine)                   |
| `StuffDocumentsChain`               | ❌ No                  | Only `answer`                                    | **Stable**             | [🔗 Docs](https://docs.langchain.com/docs/modules/chains/document/stuff)                    |


In [1]:
import warnings
warnings.filterwarnings('ignore')

from langchain_core.language_models.fake import FakeListLLM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

# Load API key
load_dotenv()
if os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# LLM
# Using FakeListLLM to bypass API Key Authentication errors.
llm = FakeListLLM(responses=['This is a mocked response'])

# Embeddings
# Using HuggingFace embeddings locally which is free and requires no API key
embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Load file
with open('sample.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Split text
splitter = CharacterTextSplitter(
    separator='\n',
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_text(raw_text)
documents = [Document(page_content=t) for t in texts]

# Vector DB
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

print('Base setup complete')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base setup complete


In [2]:
#✅ 4. StuffDocumentsChain
#Use case: Combines all docs into a single string before passing to LLM (best for small number of documents).

from langchain_classic.chains import StuffDocumentsChain
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.llm import LLMChain

# Define prompt
prompt = PromptTemplate.from_template(
    "Use the following context to answer the question:\n\n{context}\n\nQuestion: {question}"
)

# Inner LLM Chain
llm_chain = LLMChain(llm=llm, prompt=prompt)

# Stuff Chain
stuff_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_variable_name="context"
)

# Run
response = stuff_chain.invoke({
    "input_documents": documents[:3],  # test on 3 docs
    "question": "What is this document about?"
})

print("\n📌 StuffDocumentsChain Answer:", response["output_text"])


📌 StuffDocumentsChain Answer: This is a mocked response


C:\Users\premchandar\AppData\Local\Temp\ipykernel_5836\1259893393.py:14: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=prompt)
C:\Users\premchandar\AppData\Local\Temp\ipykernel_5836\1259893393.py:17: LangChainDeprecationWarning: The class `StuffDocumentsChain` was deprecated in LangChain 0.2.13 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  stuff_chain = StuffDocumentsChain(


In [3]:
# Initial prompt to summarize the first chunk
#Use case: Starts with a base answer and refines it using subsequent documents — good when each chunk contributes incrementally.

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.runnables import Runnable
from langchain_core.output_parsers import StrOutputParser

initial_prompt = PromptTemplate.from_template("""
Write a concise summary of the following text:

{context}
""")

# Refine prompt to update the previous summary with new context
refine_prompt = PromptTemplate.from_template("""
We have an existing summary:
"{existing_answer}"

Refine the summary with this new context:
"{context}"

If the context isn't useful, return the original summary.
""")

# Set up individual chains
initial_summary_chain = initial_prompt | llm | StrOutputParser()
refine_summary_chain = refine_prompt | llm | StrOutputParser()

# Start with first chunk
summary = initial_summary_chain.invoke({"context": documents[0].page_content})

# Iteratively refine with remaining docs
for doc in documents[1:]:
    summary = refine_summary_chain.invoke({
        "existing_answer": summary,
        "context": doc.page_content
    })

# Output final summary
print("📄 Refined Summary:\n")
print(summary)


📄 Refined Summary:

This is a mocked response
